# Hobby2Vec — recommending new hobbies from what people already love

**Day 2 NLP project (YDL 2026).** The brief: gather text *I* care about, turn it into
vectors with a technique from today, and make one artifact that tells a story.

**My idea — the X2Vec trick on hobbies.** Word2Vec doesn't care that its tokens are words:
any sequence works. So instead of words in a sentence, I use the **set of hobby subreddits a
person is active in** as a "sentence". Hobbies that the same people enjoy end up with similar
vectors — and that gives me:

- a **recommender**: "you like hiking and camping → try *cycling* / *Fishing*"
- an **interactive map** of hobbies where distance = how often the same people share them.

**Data:** real Reddit activity from ~40 hobby subreddits, gathered via the free
[Arctic Shift API](https://arctic-shift.photon-reddit.com) and cached to disk.

Pipeline: **gather → build sentences → train Word2Vec → inspect neighbors → recommend → map.**

In [1]:
import json, os, time, urllib.request, urllib.parse
import numpy as np
from gensim.models import Word2Vec

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("ready")

ready


## 1. Gather the data (Reddit co-occurrence)

For each hobby subreddit I page through its recent comments and collect the distinct
commenters, then invert that into **`author → the hobby subreddits they appear in`**.

The fetch is wrapped so it only hits the network **once**: if the cache file exists we load it
from disk (the brief's "cache locally" rule), otherwise we fetch (~10–15 min) and save it.

In [2]:
# ~40 hobby subreddits grouped by category (category colours the map later)
SUBS = {
    "hiking": "outdoors", "camping": "outdoors", "backpacking": "outdoors", "climbing": "outdoors",
    "running": "outdoors", "cycling": "outdoors", "Fishing": "outdoors", "birding": "outdoors",
    "Fitness": "fitness", "bodyweightfitness": "fitness", "yoga": "fitness",
    "knitting": "crafts", "crochet": "crafts", "sewing": "crafts", "woodworking": "crafts",
    "pottery": "crafts", "Embroidery": "crafts",
    "Guitar": "music", "piano": "music", "Drums": "music", "WeAreTheMusicMakers": "music",
    "chess": "games", "boardgames": "games", "DnD": "games", "magicTCG": "games",
    "drawing": "art", "painting": "art", "photography": "art", "writing": "art", "calligraphy": "art",
    "Cooking": "cooking", "Baking": "cooking", "Coffee": "cooking", "Homebrewing": "cooking", "Breadit": "cooking",
    "3Dprinting": "maker", "arduino": "maker", "electronics": "maker",
    "gardening": "green", "Aquariums": "green", "Astronomy": "green",
}

API = "https://arctic-shift.photon-reddit.com/api/comments/search"
# a real User-Agent is required - the API 403s the default "Python-urllib/x.y"
HEADERS = {"User-Agent": "ydl-2026-student-project/1.0 (educational use)"}

def fetch_authors(sub, pages=18, sleep=0.4):
    "page backwards through a subreddit's recent comments, return its distinct commenters"
    authors, before = set(), None
    for _ in range(pages):
        params = {"subreddit": sub, "limit": 100, "fields": "author,created_utc"}
        if before:
            params["before"] = before
        req = urllib.request.Request(API + "?" + urllib.parse.urlencode(params), headers=HEADERS)
        rows = json.load(urllib.request.urlopen(req, timeout=60)).get("data") or []
        if not rows:
            break
        for r in rows:
            a = r.get("author")
            if a and a not in ("[deleted]", "AutoModerator"):
                authors.add(a)
        before = min(r["created_utc"] for r in rows)   # oldest in batch -> next page
        time.sleep(sleep)                              # be polite between requests
    return authors

CACHE = "data/author_subs.json"
if os.path.exists(CACHE):
    author_subs = json.load(open(CACHE))                       # already fetched -> work from disk
    print("loaded cache:", len(author_subs), "authors")
else:
    sub_authors = {}
    for i, sub in enumerate(SUBS, 1):
        sub_authors[sub] = fetch_authors(sub)
        print(f"[{i:2d}/{len(SUBS)}] {sub:20s} {len(sub_authors[sub])} authors")
    author_subs = {}
    for sub, auth in sub_authors.items():
        for a in auth:
            author_subs.setdefault(a, []).append(sub)
    author_subs = {a: sorted(s) for a, s in author_subs.items()}
    json.dump(author_subs, open(CACHE, "w"))
    print("fetched & cached:", len(author_subs), "authors")

loaded cache: 74630 authors


## 2. Build the "sentences"

A person is only useful if they connect two hobbies, so each **sentence** is the set of hobby
subreddits of someone active in **at least two** of them. These cross-hobby people are the
co-occurrence signal Word2Vec learns from.

In [3]:
# each "sentence" = the hobby subreddits one (cross-hobby) person is active in
sentences = [subs for subs in author_subs.values() if len(subs) >= 2]
print("total authors gathered:        ", len(author_subs))
print("cross-hobby authors (sentences):", len(sentences))
print("avg hobbies per such person:    ", round(np.mean([len(s) for s in sentences]), 2))
print("example sentences:")
for s in sentences[:5]:
    print("  ", s)

total authors gathered:         74630
cross-hobby authors (sentences): 3029
avg hobbies per such person:     2.1
example sentences:
   ['backpacking', 'hiking']
   ['camping', 'hiking']
   ['backpacking', 'hiking']
   ['camping', 'hiking']
   ['Cooking', 'boardgames', 'hiking']


## 3. Train Hobby2Vec

This is the exact `Word2Vec` from the toolbox lab — the only twist is that the tokens are
**subreddits, not words**. `sg=1` is skip-gram (better on small data); the window is wide
enough to span a whole person's hobby set.

In [4]:
# X2Vec: a person's subreddit set is treated exactly like a sentence of tokens
hobby2vec = Word2Vec(sentences=sentences, vector_size=64, window=10,
                     min_count=1, sg=1, epochs=300, workers=1, seed=42)

vocab = [s for s in SUBS if s in hobby2vec.wv]   # hobbies that got a vector
print("trained vectors for", len(vocab), "of", len(SUBS), "hobby subreddits")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


trained vectors for 41 of 41 hobby subreddits


## 4. Do the neighbors make sense?

The sanity check: a hobby's nearest neighbors by cosine should be hobbies a real person might
plausibly share. No category labels were ever given to the model — it only saw who-posts-where.

In [5]:
def neighbors(sub, topn=5):
    "nearest hobby subreddits to `sub` by cosine similarity"
    return [w for w, _ in hobby2vec.wv.most_similar(sub, topn=topn)]

for s in ["hiking", "knitting", "Guitar", "3Dprinting", "Cooking", "Astronomy", "chess"]:
    print(f"{s:14s} -> {neighbors(s)}")

hiking         -> ['sewing', 'Astronomy', 'climbing', 'cycling', 'Homebrewing']
knitting       -> ['Embroidery', 'sewing', 'yoga', 'crochet', 'Baking']
Guitar         -> ['WeAreTheMusicMakers', 'Drums', 'climbing', '3Dprinting', 'photography']
3Dprinting     -> ['arduino', 'climbing', 'electronics', 'woodworking', 'Homebrewing']
Cooking        -> ['gardening', 'birding', 'Embroidery', 'piano', 'crochet']
Astronomy      -> ['climbing', 'Aquariums', 'woodworking', 'hiking', 'magicTCG']
chess          -> ['DnD', 'WeAreTheMusicMakers', 'magicTCG', 'Breadit', 'running']


## 5. The recommender

The payoff. Given the hobbies a person already enjoys, average their vectors and return the
nearest hobbies they **don't follow yet** — "new hobbies for you".

In [6]:
def recommend_hobbies(liked, topn=5):
    "given subreddits a person enjoys, suggest similar ones they don't follow yet"
    liked = [h for h in liked if h in hobby2vec.wv]
    if not liked:
        return []
    ranked = hobby2vec.wv.most_similar(positive=liked, topn=topn + len(liked))
    return [(h, round(score, 3)) for h, score in ranked if h not in liked][:topn]

for likes in [["hiking", "camping"], ["knitting", "sewing"], ["Guitar", "piano"],
              ["chess", "boardgames"], ["3Dprinting", "electronics"], ["Cooking", "Baking"]]:
    print(f"likes {str(likes):34s} -> try {recommend_hobbies(likes)}")

likes ['hiking', 'camping']              -> try [('climbing', 0.572), ('cycling', 0.547), ('sewing', 0.492), ('bodyweightfitness', 0.446), ('photography', 0.442)]
likes ['knitting', 'sewing']             -> try [('Embroidery', 0.799), ('crochet', 0.751), ('pottery', 0.665), ('Baking', 0.623), ('yoga', 0.622)]
likes ['Guitar', 'piano']                -> try [('Drums', 0.639), ('WeAreTheMusicMakers', 0.562), ('Coffee', 0.521), ('Fishing', 0.478), ('Cooking', 0.451)]
likes ['chess', 'boardgames']            -> try [('Aquariums', 0.604), ('DnD', 0.566), ('WeAreTheMusicMakers', 0.54), ('writing', 0.511), ('Breadit', 0.483)]
likes ['3Dprinting', 'electronics']      -> try [('Homebrewing', 0.641), ('arduino', 0.61), ('climbing', 0.596), ('woodworking', 0.54), ('photography', 0.529)]
likes ['Cooking', 'Baking']              -> try [('crochet', 0.626), ('sewing', 0.61), ('Embroidery', 0.599), ('pottery', 0.579), ('piano', 0.57)]


## 6. The artifact: an interactive map of hobbies — PCA vs t-SNE

Same vectors, same colours (the groups KMeans discovered), **two projections side by side** so
you can compare them yourself:

- **PCA** (left) — linear, keeps *global* distances; stable with few points.
- **t-SNE** (right) — non-linear, optimises *local* neighbourhoods; superb on big data but
  jittery on only ~40 points, so look-alikes can drift apart.

Each point is a hobby, colour = the "persona" the model found. Saved as standalone interactive HTML.

In [7]:
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

X  = np.array([hobby2vec.wv[s] for s in vocab])
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)        # unit vectors -> cosine-style clustering

# let the model's own vectors define the groups (no hand labels)
K  = 6
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Xn)

def typical(c):
    "the two hobbies closest to a cluster's centre - its most representative members"
    members = [i for i in range(len(vocab)) if km.labels_[i] == c]
    members.sort(key=lambda i: float(Xn[i] @ km.cluster_centers_[c]), reverse=True)
    return [vocab[i] for i in members[:2]]

# human-readable label per cluster, anchored to a hobby unique to that cluster
# (anchor-based so the labels survive small changes to the data / K)
ANCHORS = [
    ("Fitness",     "💪 Fitness & wellness"),
    ("backpacking", "🏕 Camping & trails"),
    ("3Dprinting",  "🔧 Maker & science"),
    ("magicTCG",    "🎲 Tabletop & geek-craft"),
    ("knitting",    "🧶 Handmade & home"),
    ("Guitar",      "🎸 Active & creative"),
]

def human_label(c):
    members = {vocab[i] for i in range(len(vocab)) if km.labels_[i] == c}
    for anchor, label in ANCHORS:
        if anchor in members:
            return label
    return " · ".join(typical(c))          # fallback: auto-name from the 2 central hobbies

label_of   = {c: human_label(c) for c in range(K)}
typical_of = {c: " · ".join(typical(c)) for c in range(K)}
group      = [label_of[c] for c in km.labels_]
typ        = [typical_of[c] for c in km.labels_]

print("discovered groups (human label  <-  most typical hobbies):")
for c in range(K):
    print(f"  {label_of[c]:24s} <- {typical_of[c]:22s}: {[vocab[i] for i in range(len(vocab)) if km.labels_[i] == c]}")

# two projections of the SAME vectors, coloured by the SAME discovered groups
pca_xy  = PCA(n_components=2, random_state=0).fit_transform(X)
tsne_xy = TSNE(n_components=2, perplexity=8, init="pca",
               learning_rate="auto", random_state=0).fit_transform(X)

rows = []
for method, xy in [("PCA", pca_xy), ("t-SNE", tsne_xy)]:
    for i, s in enumerate(vocab):
        rows.append({"method": method, "x": float(xy[i, 0]), "y": float(xy[i, 1]),
                     "hobby": s, "group": group[i], "typical": typ[i]})
dfmap = pd.DataFrame(rows)

fig = px.scatter(dfmap, x="x", y="y", color="group", text="hobby", facet_col="method",
                 title="Hobby2Vec map — PCA vs t-SNE (same vectors, colour = group the model discovered)",
                 hover_name="hobby",
                 hover_data={"x": False, "y": False, "group": True, "typical": True})
fig.update_traces(textposition="top center", marker=dict(size=10, line=dict(width=0.5, color="white")))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))   # "method=PCA" -> "PCA"
fig.update_layout(width=1200, height=640, legend_title_text="discovered group", plot_bgcolor="#fafafa")
fig.update_xaxes(matches=None, visible=False)
fig.update_yaxes(matches=None, visible=False)
fig.write_html("outputs/hobby_map.html")
print("\nsaved -> outputs/hobby_map.html  (PCA vs t-SNE side by side)")
fig.show()

discovered groups (human label  <-  most typical hobbies):
  boardgames · writing     <- boardgames · writing  : ['boardgames', 'writing']
  🏕 Camping & trails       <- climbing · yoga       : ['hiking', 'backpacking', 'climbing', 'running', 'cycling', 'yoga', 'chess']
  🔧 Maker & science        <- arduino · 3Dprinting  : ['magicTCG', 'photography', 'Homebrewing', '3Dprinting', 'arduino', 'electronics', 'Aquariums', 'Astronomy']
  💪 Fitness & wellness     <- Coffee · Fitness      : ['Fitness', 'bodyweightfitness', 'piano', 'calligraphy', 'Coffee']
  🎸 Active & creative      <- Guitar · WeAreTheMusicMakers: ['Fishing', 'Guitar', 'Drums', 'WeAreTheMusicMakers']
  🧶 Handmade & home        <- crochet · sewing      : ['camping', 'birding', 'knitting', 'crochet', 'sewing', 'woodworking', 'pottery', 'Embroidery', 'DnD', 'drawing', 'painting', 'Cooking', 'Baking', 'Breadit', 'gardening']



saved -> outputs/hobby_map.html  (PCA vs t-SNE side by side)


## 7. The fun artifact: a "Hobby Matchmaker" game 🎯

The map is nice to look at, but the real payoff is *playable*. This cell bakes the trained
Hobby2Vec vectors into a **standalone interactive HTML page**: you tick the hobbies you already
enjoy, hit a button, and it recommends a new one — with a match score and a "because you like…"
explanation.

The recommender logic runs **client-side in JavaScript** (it re-implements `most_similar`:
average the chosen hobby vectors, then rank the rest by cosine), so the page is fully offline —
just open `outputs/hobby_matchmaker.html` in any browser.

In [8]:
import json

# bake the trained vectors + discovered groups into a small JSON payload for the page
vecs = {}
for s in vocab:
    v = hobby2vec.wv[s].astype(float)
    v = v / (np.linalg.norm(v) + 1e-9)          # unit vectors -> dot product == cosine in JS
    vecs[s] = [round(float(x), 4) for x in v]

group_of     = {vocab[i]: group[i] for i in range(len(vocab))}      # human label per hobby
groups_order = list(dict.fromkeys(group))                            # unique, keeps order
palette      = ["#6C8EF5", "#E8743B", "#1DBC9B", "#E0529C", "#9B5DE5", "#F2B705", "#3CA0A0"]
group_color  = {g: palette[i % len(palette)] for i, g in enumerate(groups_order)}

DATA = {"vecs": vecs, "group_of": group_of, "groups": groups_order, "group_color": group_color}

PAGE = r'''<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Hobby Matchmaker</title>
<style>
  :root { --card:#ffffffee; }
  * { box-sizing:border-box; }
  body { margin:0; min-height:100vh; font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;
         color:#1f2330; background:linear-gradient(135deg,#6a8dff 0%,#9b5de5 50%,#ff7eb3 100%);
         background-attachment:fixed; padding:32px 16px; }
  .wrap { max-width:920px; margin:0 auto; }
  h1 { text-align:center; color:#fff; margin:4px 0 2px; font-size:34px; text-shadow:0 2px 14px #0003; }
  .sub { text-align:center; color:#ffffffdd; margin:0 0 22px; font-size:15px; }
  .panel { background:var(--card); border-radius:20px; padding:22px 22px 26px;
           box-shadow:0 18px 50px #0000003a; backdrop-filter:blur(6px); }
  .grp { margin:14px 0 6px; font-weight:700; font-size:14px; display:flex; align-items:center; gap:8px; }
  .dot { width:11px; height:11px; border-radius:50%; display:inline-block; }
  .chips { display:flex; flex-wrap:wrap; gap:9px; }
  .chip { padding:8px 13px; border-radius:999px; border:2px solid #e6e6ee; background:#fff;
          cursor:pointer; font-size:13.5px; user-select:none; transition:.16s; }
  .chip:hover { transform:translateY(-2px); box-shadow:0 6px 16px #0002; }
  .chip.on { color:#fff; border-color:transparent; transform:translateY(-1px); box-shadow:0 6px 16px #0003; }
  .bar { display:flex; gap:12px; justify-content:center; margin-top:22px; flex-wrap:wrap; }
  button { border:0; border-radius:999px; padding:13px 26px; font-size:15px; font-weight:700;
           cursor:pointer; transition:.16s; }
  .go { background:#1f2330; color:#fff; }
  .go:hover { transform:translateY(-2px); box-shadow:0 10px 24px #0004; }
  .ghost { background:#eef0f6; color:#444; }
  .count { text-align:center; color:#666; font-size:13px; margin-top:10px; min-height:18px; }
  #result { margin-top:18px; }
  .hero { background:linear-gradient(135deg,#1f2330,#3a3f55); color:#fff; border-radius:18px;
          padding:22px 24px; text-align:center; animation:pop .45s cubic-bezier(.2,1.3,.4,1); }
  .hero .big { font-size:30px; font-weight:800; margin:6px 0; }
  .hero .pct { font-size:14px; opacity:.9; }
  .hero .why { font-size:13.5px; opacity:.85; margin-top:8px; }
  .runners { display:flex; gap:12px; margin-top:12px; flex-wrap:wrap; justify-content:center; }
  .mini { flex:1; min-width:170px; background:#fff; border-radius:14px; padding:14px 16px;
          box-shadow:0 6px 18px #0000001f; animation:pop .5s cubic-bezier(.2,1.3,.4,1); }
  .mini .nm { font-weight:700; font-size:16px; }
  .mini .gp { font-size:12px; color:#777; margin-top:2px; }
  .mini .pb { height:7px; border-radius:6px; background:#eee; margin-top:9px; overflow:hidden; }
  .mini .pf { height:100%; border-radius:6px; }
  @keyframes pop { from { opacity:0; transform:scale(.9) translateY(8px);} to {opacity:1; transform:none;} }
</style></head>
<body><div class="wrap">
  <h1>🎯 Hobby Matchmaker</h1>
  <p class="sub">Tick the hobbies you already love — Hobby2Vec will find your next obsession.</p>
  <div class="panel">
    <div id="board"></div>
    <div class="bar">
      <button class="go" onclick="recommend()">✨ Find my next hobby</button>
      <button class="ghost" onclick="surprise()">🎲 Surprise me</button>
      <button class="ghost" onclick="clearAll()">Clear</button>
    </div>
    <div class="count" id="count"></div>
    <div id="result"></div>
  </div>
</div>
<script>
const DATA = __DATA__;
const hobbies = Object.keys(DATA.vecs);
const DIM = DATA.vecs[hobbies[0]].length;
const selected = new Set();
const emojiOf = g => (DATA.group_of ? g.split(' ')[0] : '');

function render(){
  const board = document.getElementById('board');
  board.innerHTML = '';
  DATA.groups.forEach(g => {
    const head = document.createElement('div'); head.className = 'grp';
    head.innerHTML = '<span class="dot" style="background:'+DATA.group_color[g]+'"></span>'+g;
    const chips = document.createElement('div'); chips.className = 'chips';
    hobbies.filter(h => DATA.group_of[h] === g).forEach(h => {
      const c = document.createElement('div'); c.className = 'chip'; c.textContent = h;
      if (selected.has(h)) { c.classList.add('on'); c.style.background = DATA.group_color[g]; }
      c.onclick = () => { selected.has(h) ? selected.delete(h) : selected.add(h); render(); updateCount(); };
      chips.appendChild(c);
    });
    board.appendChild(head); board.appendChild(chips);
  });
}
function updateCount(){
  document.getElementById('count').textContent =
    selected.size ? selected.size + ' selected' : 'Pick at least one hobby to get a recommendation.';
}
function recommend(){
  const res = document.getElementById('result');
  if (selected.size === 0){ res.innerHTML = ''; updateCount(); return; }
  const q = new Array(DIM).fill(0);
  selected.forEach(h => { const v = DATA.vecs[h]; for (let i=0;i<DIM;i++) q[i]+=v[i]; });
  let n=0; for (let i=0;i<DIM;i++) n+=q[i]*q[i]; n=Math.sqrt(n)||1;
  for (let i=0;i<DIM;i++) q[i]/=n;
  const scored = hobbies.filter(h => !selected.has(h)).map(h => {
    const v = DATA.vecs[h]; let d=0; for (let i=0;i<DIM;i++) d+=q[i]*v[i];
    return { h, s:d };
  }).sort((a,b)=>b.s-a.s);
  if (!scored.length){ res.innerHTML = '<p>Pick fewer hobbies to leave something to suggest!</p>'; return; }
  const top = scored.slice(0,3);
  // explanation: the 2 selected hobbies most similar to the winner
  const win = top[0].h, wv = DATA.vecs[win];
  const why = [...selected].map(h => { const v=DATA.vecs[h]; let d=0; for(let i=0;i<DIM;i++) d+=wv[i]*v[i]; return {h,d}; })
                .sort((a,b)=>b.d-a.d).slice(0,2).map(x=>x.h);
  const pct = s => Math.max(1, Math.round(s*100));
  let html = '<div class="hero">your next obsession could be…'
           + '<div class="big">'+emojiOf(DATA.group_of[win])+' '+win+'</div>'
           + '<div class="pct">'+pct(top[0].s)+'% match · '+DATA.group_of[win]+'</div>'
           + '<div class="why">because you like <b>'+why.join('</b> & <b>')+'</b></div></div>';
  if (top.length>1){
    html += '<div class="runners">';
    top.slice(1).forEach(t => {
      const col = DATA.group_color[DATA.group_of[t.h]];
      html += '<div class="mini"><div class="nm">'+emojiOf(DATA.group_of[t.h])+' '+t.h+'</div>'
            + '<div class="gp">'+DATA.group_of[t.h]+'</div>'
            + '<div class="pb"><div class="pf" style="width:'+pct(t.s)+'%;background:'+col+'"></div></div>'
            + '<div class="gp" style="margin-top:6px">'+pct(t.s)+'% match</div></div>';
    });
    html += '</div>';
  }
  res.innerHTML = html;
}
function clearAll(){ selected.clear(); render(); updateCount(); document.getElementById('result').innerHTML=''; }
function surprise(){
  selected.clear();
  const pool = hobbies.slice(); for (let i=pool.length-1;i>0;i--){ const j=Math.floor(Math.random()*(i+1)); [pool[i],pool[j]]=[pool[j],pool[i]]; }
  pool.slice(0, 3).forEach(h => selected.add(h));
  render(); updateCount(); recommend();
}
render(); updateCount();
</script></body></html>'''

html = PAGE.replace("__DATA__", json.dumps(DATA))
with open("outputs/hobby_matchmaker.html", "w") as f:
    f.write(html)
print("saved -> outputs/hobby_matchmaker.html  (", len(html), "bytes )")

saved -> outputs/hobby_matchmaker.html  ( 32087 bytes )


## What I learned

- **The X2Vec trick really works on real data.** With no labels at all, Word2Vec recovered
  sensible hobby families purely from *who posts where*: knitting sits next to crochet,
  embroidery and pottery; hiking next to camping and cycling; 3D-printing next to arduino and
  electronics.
- **Co-occurrence = recommendation.** The same vectors that draw the map also power a working
  "try this next" recommender — one embedding space, two artifacts.
- **Data quality dominates.** My first attempt (a hobby *survey*, people rating interests 1-5)
  gave noise, because "rates everything highly" swamps the signal. Switching to **behavioural**
  data (people actually showing up in multiple hobby communities) made the structure appear.
- **The visualization choice matters as much as the model.** t-SNE scattered similar hobbies
  because it is unreliable on only ~40 points, and my hand-made categories disagreed with the
  data. Switching to **PCA** and letting **KMeans discover the groups** made the map honest:
  the colours are the hobby "personas" the vectors actually contain, not my taxonomy.
- **Limits:** loner hobbies with few cross-over users (e.g. chess) have noisier neighbors, and
  the map only knows the ~40 subreddits I sampled.